# Chapter 11: Segmentation done right

In [1]:
import numpy as np
import pandas as pd
from expkit.segments.behavioral import simulate_population, BEHAVIORAL_LABELS
from expkit.plot.style import apply_style
apply_style()

## Population breakdown

In [2]:
df = simulate_population(5000, seed=110)
print(df['segment'].value_counts())
print('axis means by segment:')
print(df.groupby('segment')[['weekly_active_rate', 'contribution_rate', 'intentional_rate']].mean().round(3))

segment
passive_consumer      1898
silent_intentional    1249
active_contributor    1228
active_consumer        625
Name: count, dtype: int64
axis means by segment:
                    weekly_active_rate  contribution_rate  intentional_rate
segment                                                                    
active_consumer                  0.414              0.072             0.474
active_contributor               0.409              0.241             0.334
passive_consumer                 0.245              0.127             0.189
silent_intentional               0.159              0.154             0.476


## Demographic vs behavioural slicing

In [3]:
rng = np.random.default_rng(110)
n = 5000
df = simulate_population(n, seed=110)
df['country'] = rng.choice(['A', 'B', 'C'], size=n)
df['arm'] = rng.choice(['control', 'treatment'], size=n)
treat_lift = {'active_contributor': 0.10, 'active_consumer': 0.04, 'silent_intentional': -0.03, 'passive_consumer': 0.0}
df['outcome'] = 0
base = 0.30
for seg, l in treat_lift.items():
    for arm in ['control', 'treatment']:
        m = (df['segment'] == seg) & (df['arm'] == arm)
        p = base + (l if arm == 'treatment' else 0.0)
        df.loc[m, 'outcome'] = rng.binomial(1, max(0, min(1, p)), size=int(m.sum()))
print('--- by country ---')
print(df.groupby(['country', 'arm'])['outcome'].mean().unstack().round(3))
print('\n--- by behavioural segment ---')
print(df.groupby(['segment', 'arm'])['outcome'].mean().unstack().round(3))
print('\nPooled treatment - control =', round(df.groupby('arm')['outcome'].mean().diff().iloc[-1], 3))

--- by country ---
arm      control  treatment
country                    
A          0.315      0.343
B          0.296      0.352
C          0.316      0.329

--- by behavioural segment ---
arm                 control  treatment
segment                               
active_consumer       0.261      0.390
active_contributor    0.354      0.420
passive_consumer      0.299      0.329
silent_intentional    0.302      0.262

Pooled treatment - control = 0.033
